# Experiment: H5 QC Viewer

Notebook para inspeccionar rapidamente archivos `.h5` del pipeline (inputs o predicciones):
- datasets/keys disponibles,
- shapes y tipos,
- estadisticas basicas,
- visualizacion de slices y MIP.


In [ ]:
from __future__ import annotations

from pathlib import Path
import glob
import numpy as np
import h5py
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"


In [ ]:
# Ajusta estos paths segun lo que quieras revisar
H5_GLOB = "../data/paired_dataset/h5_lr/*.h5"   # ej: inputs H5
# H5_GLOB = "../data/predictions_h5_3t_gpu/*.h5"  # ej: predicciones H5

h5_files = sorted(glob.glob(H5_GLOB))
print(f"Found {len(h5_files)} files")
for fp in h5_files[:10]:
    print(" -", Path(fp).name)


In [ ]:
def summarize_h5(path: str):
    print(f"\n=== {Path(path).name} ===")
    with h5py.File(path, "r") as f:
        keys = list(f.keys())
        print("keys:", keys)
        for k in keys:
            obj = f[k]
            if isinstance(obj, h5py.Dataset):
                print(f"{k:15s} shape={obj.shape} dtype={obj.dtype}")


def dataset_stats(path: str, key: str, frame: int = 0):
    with h5py.File(path, "r") as f:
        if key not in f:
            print(f"Key '{key}' not found")
            return
        arr = np.asarray(f[key])
        if arr.ndim >= 4:
            arr = arr[frame]
        elif arr.ndim >= 1 and arr.shape[0] > 1:
            arr = arr[frame]
        arr = np.asarray(arr)
        print(f"{key}: shape={arr.shape}, min={arr.min():.6g}, max={arr.max():.6g}, mean={arr.mean():.6g}, std={arr.std():.6g}")


In [ ]:
# Resumen rapido de los primeros archivos
for fp in h5_files[:3]:
    summarize_h5(fp)


In [ ]:
# Cambia el indice para inspeccionar otro archivo
case_idx = 0
fp = h5_files[case_idx]

for k in ["u", "v", "w", "mag_u", "mag_v", "mag_w", "mask"]:
    dataset_stats(fp, k, frame=0)


In [ ]:
def get_3d_volume(f: h5py.File, key: str, frame: int = 0):
    arr = np.asarray(f[key])
    # Expected common formats: [T,X,Y,Z] or [X,Y,Z]
    if arr.ndim == 4:
        return arr[frame]
    if arr.ndim == 3:
        return arr
    raise ValueError(f"Unsupported ndim for key {key}: {arr.ndim}")


def plot_mid_slices(vol: np.ndarray, title: str):
    x, y, z = vol.shape
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(vol[x // 2, :, :].T, origin="lower")
    ax[0].set_title("sagittal mid")
    ax[1].imshow(vol[:, y // 2, :].T, origin="lower")
    ax[1].set_title("coronal mid")
    ax[2].imshow(vol[:, :, z // 2].T, origin="lower")
    ax[2].set_title("axial mid")
    fig.suptitle(title)
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()


def plot_mip(vol: np.ndarray, title: str):
    mip = np.max(vol, axis=2)
    plt.figure(figsize=(6, 6))
    plt.imshow(mip.T, origin="lower")
    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
# Visualizacion de un caso
case_idx = 0
frame = 0
key = "u"   # prueba: u, v, w, mag_u, mask

fp = h5_files[case_idx]
with h5py.File(fp, "r") as f:
    vol = get_3d_volume(f, key=key, frame=frame)

print(Path(fp).name, key, vol.shape)
plot_mid_slices(vol, f"{Path(fp).name} | {key} | frame={frame}")
plot_mip(vol, f"MIP axial | {Path(fp).name} | {key}")


## Notas

- Si el H5 viene de `nifti_to_h5`, normalmente veras keys como `u,v,w,mag_u,mag_v,mag_w,mask,venc_*,dx,triggerTimes`.
- Para predicciones, las keys pueden cambiar segun el script de guardado.
- Si quieres comparar LR vs prediccion, carga ambos archivos y grafica mismos `frame`/slices lado a lado.


## Comparacion LowRes (H5) vs HighRes 7T (NIfTI)

Esta seccion compara un caso del `.h5` (LR) contra su contraparte NIfTI 7T (HR/ground truth).
Asume que el nombre del archivo H5 es el `case_id` (por ejemplo `001_20240313.h5`) y que existe una carpeta
`<HR_NIFTI_ROOT>/<case_id>/` con `Vx.nii.gz`, `Vy.nii.gz`, `Vz.nii.gz`.


In [ ]:
import nibabel as nib
from pathlib import Path

# Ajusta estos paths a tu estructura
HR_NIFTI_ROOT = Path("../data/paired_dataset/hr_7t_in_3t")
HR_KEYS = {"u": "Vx.nii.gz", "v": "Vy.nii.gz", "w": "Vz.nii.gz"}


def case_id_from_h5(path: str) -> str:
    return Path(path).stem


def load_h5_component(path: str, key: str, frame: int = 0):
    with h5py.File(path, "r") as f:
        if key not in f:
            raise KeyError(f"Key '{key}' not found in {path}")
        arr = np.asarray(f[key])
    if arr.ndim == 4:   # [T,X,Y,Z]
        frame = min(frame, arr.shape[0] - 1)
        return arr[frame], frame
    if arr.ndim == 3:
        return arr, 0
    raise ValueError(f"Unsupported H5 shape for key {key}: {arr.shape}")


def load_nifti_component(path: Path, frame: int = 0):
    if not path.exists():
        raise FileNotFoundError(path)
    img = nib.load(str(path))
    arr = np.asarray(img.dataobj)
    if arr.ndim == 4:   # expected [X,Y,Z,T]
        frame = min(frame, arr.shape[-1] - 1)
        return arr[..., frame], frame
    if arr.ndim == 3:
        return arr, 0
    raise ValueError(f"Unsupported NIfTI shape: {arr.shape}")


def summarize_pair(lr_vol: np.ndarray, hr_vol: np.ndarray, tag: str):
    print(f"{tag} LR shape={lr_vol.shape} | HR shape={hr_vol.shape}")
    print(f"{tag} LR min/max/mean/std = {lr_vol.min():.6g}, {lr_vol.max():.6g}, {lr_vol.mean():.6g}, {lr_vol.std():.6g}")
    print(f"{tag} HR min/max/mean/std = {hr_vol.min():.6g}, {hr_vol.max():.6g}, {hr_vol.mean():.6g}, {hr_vol.std():.6g}")


In [ ]:
def show_pair_slices(lr_vol: np.ndarray, hr_vol: np.ndarray, title: str):
    # Use center slice of each volume independently (works even if shapes differ)
    z_lr = lr_vol.shape[2] // 2
    z_hr = hr_vol.shape[2] // 2

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(lr_vol[:, :, z_lr].T, origin="lower")
    ax[0].set_title("LowRes H5 (axial mid)")
    ax[0].axis("off")

    ax[1].imshow(hr_vol[:, :, z_hr].T, origin="lower")
    ax[1].set_title("HighRes 7T NIfTI (axial mid)")
    ax[1].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_pair_mip(lr_vol: np.ndarray, hr_vol: np.ndarray, title: str):
    mip_lr = np.max(lr_vol, axis=2)
    mip_hr = np.max(hr_vol, axis=2)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(mip_lr.T, origin="lower")
    ax[0].set_title("LowRes H5 MIP")
    ax[0].axis("off")

    ax[1].imshow(mip_hr.T, origin="lower")
    ax[1].set_title("HighRes 7T MIP")
    ax[1].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Selecciona caso/componente/frame para comparar
case_idx = 0              # indice dentro de h5_files
component = "u"          # "u", "v", "w"
frame = 0

fp = h5_files[case_idx]
case_id = case_id_from_h5(fp)
hr_case_dir = HR_NIFTI_ROOT / case_id
hr_path = hr_case_dir / HR_KEYS[component]

print("H5 case:", Path(fp).name)
print("HR NIfTI:", hr_path)

lr_vol, used_frame_lr = load_h5_component(fp, component, frame=frame)
hr_vol, used_frame_hr = load_nifti_component(hr_path, frame=frame)

summarize_pair(lr_vol, hr_vol, f"{case_id} | {component} | frame req={frame} (lr={used_frame_lr}, hr={used_frame_hr})")
show_pair_slices(lr_vol, hr_vol, f"{case_id} | {component} | axial")
show_pair_mip(lr_vol, hr_vol, f"{case_id} | {component} | MIP")


### Nota de interpretacion

- LR y HR pueden tener distinta resolucion o rango de intensidad (normalizacion/escala), por lo que esta comparacion es de **QC visual y estadistica basica**.
- Si quieres comparar voxel-a-voxel (error map), primero hay que re-muestrear uno al grid del otro.
